In [1]:
import pandas as pd 
import torch 
import torch.nn as nn 

In [2]:
df = pd.read_csv ( '/home/dung/machinelearning/reservoir computing/Mackey-Glass Time Series(taw17).csv')

In [3]:
df.drop ( columns= 'index',inplace = True)

In [9]:
df.shape 

(1100, 3)

AFTER THIS CELL THE CODE WAS SUPPORTED BY CLAUDE CODE

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.optim import Adam
from reservoir_computing import Reservoir_computing

def prepare_data(df: pd.DataFrame):
    """
    Chuẩn bị data từ DataFrame.
    Input features: ['t', 't-taw']
    Target: ['t+1']
    """
    features = torch.tensor(df[['t', 't-taw']].values, dtype=torch.float32)  # (T, 2)
    targets  = torch.tensor(df[['t+1']].values,         dtype=torch.float32)  # (T, 1)
    return features, targets


# ── Cách 1: Ridge Regression (KHUYẾN NGHỊ cho ESN) ──────────────────────────
def train_ridge(model: Reservoir_computing, df: pd.DataFrame, ridge_alpha: float = 1e-4):
    """
    Closed-form solution: W_out = (X^T X + alpha*I)^{-1} X^T Y
    Nhanh, ổn định, không cần epoch.
    """
    features, targets = prepare_data(df)

    # Thu thập toàn bộ augmented states (không gradient)
    model.W_out.weight.requires_grad_(False)

    with torch.no_grad():
        T = features.shape[0]
        current_state = torch.zeros(model.reservoir_dim, dtype=torch.float32)
        states_history = []

        for t in range(T):
            u_t         = features[t]
            input_part  = torch.matmul(model.W_in, u_t)
            res_part    = torch.matmul(model.Wrc, current_state)
            new_state   = ((1 - model.learning_rate) * current_state
                           + model.learning_rate * torch.tanh(input_part + res_part))
            current_state = new_state
            states_history.append(current_state)

        X_states    = torch.stack(states_history)               # (T, reservoir_dim)
        X_augmented = torch.cat([X_states, features], dim=1)   # (T, reservoir_dim + input_dim)

    # Ridge regression: W = (X^T X + αI)^{-1} X^T Y
    X = X_augmented  # (T, D)
    Y = targets      # (T, output_dim)

    I   = torch.eye(X.shape[1]) * ridge_alpha
    W   = torch.linalg.solve(X.T @ X + I, X.T @ Y)  # (D, output_dim)

    # Gán vào W_out — Linear lưu weight shape (out, in)
    model.W_out.weight = nn.Parameter(W.T, requires_grad=False)

    # Tính loss cuối
    Y_pred = X_augmented @ W
    loss   = torch.mean((Y_pred - Y) ** 2)
    print(f"[Ridge] Training MSE: {loss.item():.6f}")
    return model


# ── Cách 2: Gradient Descent (nếu muốn fine-tune W_out) ─────────────────────
def train_gradient(
    model: Reservoir_computing,
    df: pd.DataFrame,
    epochs: int = 200,
    lr: float = 1e-3,
    washout: int = 50,          # bỏ qua T bước đầu để reservoir ổn định
    print_every: int = 20,
):
    """
    Chỉ train W_out bằng gradient descent.
    Reservoir (W_in, Wrc) giữ nguyên — đúng với ESN.
    """
    features, targets = prepare_data(df)

    # Chỉ W_out được train
    optimizer = Adam(model.W_out.parameters(), lr=lr)
    criterion = nn.MSELoss()

    # Tính sẵn states (không đổi qua các epoch vì reservoir cố định)
    with torch.no_grad():
        T = features.shape[0]
        current_state = torch.zeros(model.reservoir_dim, dtype=torch.float32)
        states_history = []

        for t in range(T):
            u_t         = features[t]
            input_part  = torch.matmul(model.W_in, u_t)
            res_part    = torch.matmul(model.Wrc, current_state)
            new_state   = ((1 - model.learning_rate) * current_state
                           + model.learning_rate * torch.tanh(input_part + res_part))
            current_state = new_state
            states_history.append(current_state)

        X_states    = torch.stack(states_history)
        X_augmented = torch.cat([X_states, features], dim=1)

    # Bỏ washout period
    X_train = X_augmented[washout:]   # (T-washout, D)
    Y_train = targets[washout:]       # (T-washout, output_dim)

    loss_history = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        Y_pred = model.W_out(X_train)
        loss   = criterion(Y_pred, Y_train)
        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())
        if (epoch + 1) % print_every == 0:
            print(f"Epoch [{epoch+1:>4}/{epochs}]  MSE Loss: {loss.item():.6f}")

    print(f"\nFinal MSE: {loss_history[-1]:.6f}")
    return model, loss_history


# ── Inference ────────────────────────────────────────────────────────────────
def predict(model: Reservoir_computing, df: pd.DataFrame):
    features, targets = prepare_data(df)
    with torch.no_grad():
        Y_pred = model.forward(features)
    return Y_pred, targets